# Modal CIFAR-100 Wasserstein geometry smoke test

This notebook only orchestrates and visualizes the existing research pipeline. It runs one seed for 20 epochs on a Modal T4, using real CIFAR-100, four training anchors, 16 evaluation widths, Sliced Wasserstein, and one Euclidean-mean control. It does **not** run oracles, CFM, geometry regularization, arbitrary-width training, or hyperparameter search.

## 1. Modal setup

Install the current Modal client. Run `modal setup` once to authenticate this notebook environment.

In [ ]:
%pip install -q "modal>=1.0" pandas matplotlib scipy pyyaml

In [ ]:
import subprocess
subprocess.run(["modal", "setup"], check=True)

### Clone the research repository

Leave `GITHUB_TOKEN` empty for a public repository. If authentication is needed, paste a short-lived read-only token in the marked field. The token is passed as an in-memory HTTP header, not embedded in the URL or printed. Clear it before saving or sharing this notebook.

In [ ]:
from pathlib import Path
import base64
import sys

REPO_URL = "https://github.com/duyh80456-code/new-pruning.git"
GITHUB_TOKEN = ""  # <-- Điền GitHub token tại đây nếu repo yêu cầu xác thực
PROJECT_ROOT = (Path.cwd() / "new-pruning-modal").resolve()

if not (PROJECT_ROOT / ".git").exists():
    command = ["git"]
    if GITHUB_TOKEN:
        credential = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
        command += ["-c", f"http.extraHeader=Authorization: Basic {credential}"]
    command += ["clone", REPO_URL, str(PROJECT_ROOT)]
    subprocess.run(command, check=True)
else:
    print(f"Reusing existing clone: {PROJECT_ROOT}")

assert (PROJECT_ROOT / "modal_smoke.py").is_file(), "Clone does not contain modal_smoke.py"
sys.path.insert(0, str(PROJECT_ROOT))
GITHUB_TOKEN = ""  # remove the notebook reference as soon as cloning finishes
if "credential" in globals(): del credential
if "command" in globals(): del command
print(f"Project root: {PROJECT_ROOT}")

## 2. Smoke-test configuration

Inspect and assert the exact scientific configuration before allocating GPU time.

In [ ]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "modal_smoke.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text())
anchors = [float(x) for x in config["compression"]["train_widths"]]
eval_widths = [float(x) for x in config["compression"]["eval_widths"]]
assert config["dataset"]["name"].lower() == "cifar100"
assert config["dataset"]["fake_data"] is False
assert config["experiment"]["seeds"] == [0]
assert config["training"]["epochs"] == 20
assert anchors == [0.25, 0.50, 0.75, 1.00]
assert eval_widths == [round(0.25 + 0.05 * i, 2) for i in range(16)]
assert config["geometry"]["method"] == "sliced_wasserstein"
assert config["geometry"]["control_methods"] == ["euclidean_mean"]
assert config["oracle"]["enabled"] is False
print(yaml.safe_dump(config, sort_keys=False))

## 3. Launch remote T4 experiment

The source clone is mounted into `/workspace`; CIFAR-100, checkpoints, features, and outputs persist in the `new-pruning-smoke-data` Modal Volume. Choose a fresh run name on every launch.

In [ ]:
from datetime import datetime, timezone
import modal

from modal_smoke import app, run_smoke

RUN_NAME = datetime.now(timezone.utc).strftime("modal-smoke-seed0-%Y%m%d-%H%M%S")
print(f"Launching {RUN_NAME}")
with modal.enable_output():
    with app.run():
        result = run_smoke.remote(RUN_NAME)

## 4. Training/evaluation status

In [ ]:
from IPython.display import Markdown, display

print(f"GPU: {result['gpu']}")
print(f"Modal Volume: {result['volume_name']}")
print(f"Output path: {result['volume_output_path']}")
print(f"Report path: {result['modal_report_path']}")
display(Markdown(result["report_markdown"]))

## 5. Load compact results

Only compact tables and the 16×16 matrix are returned. Feature tensors stay on the Volume.

In [ ]:
import numpy as np
import pandas as pd

accuracy = pd.DataFrame(result["accuracy_table"])
local = pd.DataFrame(result["local_sensitivity"])
wasserstein = np.asarray(result["wasserstein_matrix"], dtype=float)
budgets = np.asarray(result["budgets"], dtype=float)
assert np.isfinite(wasserstein).all()
display(accuracy[["budget", "seen_unseen", "accuracy", "flops", "params"]])

## 6. Accuracy vs budget

In [ ]:
import matplotlib.pyplot as plt

anchors_frame = accuracy[accuracy["is_train_anchor"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(accuracy["budget"], accuracy["accuracy"], marker="o", label="all budgets")
ax.scatter(anchors_frame["budget"], anchors_frame["accuracy"], marker="*", s=130, color="crimson", label="training anchors")
ax.set(xlabel="Width budget", ylabel="Accuracy", title="CIFAR-100 accuracy vs width")
ax.grid(alpha=0.25); ax.legend(); plt.show()

## 7. Wasserstein geometry

`G(c)` is the local Sliced Wasserstein jump divided by the fixed grid step 0.05. The second panel shows `P(c)`, the corresponding absolute accuracy change divided by 0.05.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(local["budget_start"], local["G_width"], marker="o")
axes[0].set(xlabel="Interval start c", ylabel="G(c)", title="Local Wasserstein sensitivity")
axes[1].plot(local["budget_start"], local["P"], marker="o", color="darkorange")
axes[1].set(xlabel="Interval start c", ylabel="P(c)", title="Local accuracy sensitivity")
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 8. Geometry vs performance

In [ ]:
corr = result["correlations"]
fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(local["G_width"], local["P"], c=local["budget_start"], cmap="viridis", s=65)
ax.set(xlabel="Wasserstein sensitivity G(c)", ylabel="Accuracy sensitivity P(c)", title="Geometry vs performance")
ax.grid(alpha=0.25); fig.colorbar(points, ax=ax, label="Interval start c"); plt.show()
print(f"Pearson r={corr['pearson_r']}, p={corr['pearson_p_value']}")
print(f"Spearman rho={corr['spearman_rho']}, p={corr['spearman_p_value']}")
print(f"Euclidean-mean control: Spearman rho={corr['euclidean_mean_spearman_rho']}, p={corr['euclidean_mean_spearman_p_value']}")
if corr["spearman_rho"] is not None and corr["euclidean_mean_spearman_rho"] is not None:
    comparison = abs(corr["spearman_rho"]) - abs(corr["euclidean_mean_spearman_rho"])
    print(f"|rho_SW| - |rho_mean| = {comparison:.4f} (negative means the cheap mean control is at least as strong in this run)")

## 9. Pairwise Wasserstein heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(wasserstein, cmap="magma", origin="lower")
labels = [f"{x:.2f}" for x in budgets]
ax.set_xticks(range(len(labels)), labels, rotation=90)
ax.set_yticks(range(len(labels)), labels)
ax.set(xlabel="Width budget", ylabel="Width budget", title="Pairwise Sliced Wasserstein distance")
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()

## 10. GO / NO-GO summary

The label below is a descriptive multi-signal triage, not a claim that the hypothesis or paper is correct. Always inspect the displayed values and the Euclidean-mean control.

In [ ]:
display(Markdown(f"## {result['interpretation']}"))
for item in result["interpretation_evidence"]:
    print("-", item)
print(f"Compression cliff candidates: {len(result['compression_cliffs'])}")
print("This is a one-seed, short-training smoke test intended only to assess whether the research hypothesis warrants a full experiment.")

### Optional: download all persisted artifacts

The experiment is already safe on the Modal Volume. Run this cell only when checkpoint/features are needed locally; otherwise keep the compact notebook result.

In [ ]:
DOWNLOAD_FULL_ARTIFACTS = False
if DOWNLOAD_FULL_ARTIFACTS:
    local_destination = Path.cwd() / "modal-results" / RUN_NAME
    local_destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["modal", "volume", "get", result["volume_name"], result["volume_output_path"], str(local_destination)], check=True)
    print(f"Downloaded to {local_destination}")
else:
    print(f"Artifacts remain at Modal Volume {result['volume_name']}:{result['volume_output_path']}")